# Test Faulty Records
Load and analyze debug data for records with Dep_factor > 1

In [ ]:
import pandas as pd
import json
import os
from glm_model_test import load_glm_model, predict_glm

In [ ]:
# Load config
with open('config.json', 'r') as f:
    config = json.load(f)

debug_folder = config['debug_folder']
bsst_glm_path = config['BSST_GLM_path']

print(f"Debug folder: {debug_folder}")
print(f"BSST GLM path: {bsst_glm_path}")

In [ ]:
# Load debug data for a specific BSST segment
bsst = 'Entry_Level_Car_GLM'
debug_car_with_dep_factor = os.path.join(debug_folder, f'Records_{bsst}.parquet')
df_dep_fact = pd.read_parquet(debug_car_with_dep_factor)
print(f"Data loaded successfully! Shape: {df_dep_fact.shape}")
df_dep_fact.head()

In [ ]:
# Load GLM model for this BSST segment
model_path = os.path.join(bsst_glm_path, f"{bsst}.json")

with open(model_path, 'r') as f:
    glm_model = json.load(f)
    
print(f"Loaded GLM model: {glm_model['model_name']}")
print(f"Intercept: {glm_model['intercept']}")
print(f"Number of coefficients: {len(glm_model['coefficients'])}")
print(f"\nAll coefficients:")
for name, value in glm_model['coefficients'].items():
    print(f"  {name}: {value}")

In [ ]:
# Run predictions on the debug data
predictions = predict_glm(df_dep_fact, glm_model)

# Add predictions to df_dep_fact for comparison
df_dep_fact['Dep_factor_recalc'] = predictions.values

print(f"Predictions computed for {len(predictions):,} records")
print(f"\nOriginal Dep_factor stats:")
print(df_dep_fact['Dep_factor'].describe())
print(f"\nRecalculated Dep_factor stats:")
print(df_dep_fact['Dep_factor_recalc'].describe())

In [ ]:
# Compare original vs recalculated
df_dep_fact[['ODOMETER', 'geo_pop_density_ntile', 'CALC_VEH_AGE', 'Dep_factor', 'Dep_factor_recalc']].head(10)

In [ ]:
# Check all columns in the debug data
print(f"Columns: {df_dep_fact.columns.tolist()}")
print(f"\nData types:")
print(df_dep_fact.dtypes)

In [ ]:
# Summary statistics for all numeric columns
df_dep_fact.describe()